# 🎨 Zero-Trust Ephemeral Ciphertext Queue & Media Studio Worker (`v1.2.0`)

Execute image & video generation (`MOSS-Image-SFT`, `SDXL`, `FLUX.1`, `Wan 2.1/2.2 T2V & I2V`, `HunyuanVideo-1.5`) on untrusted remote GPUs with zero plaintext persistence:
1. Run **Cell 1** once to enter your session passphrase in memory (0.1s prompt before background package setup), disable IPython history persistence, and initialize the TorchAO FP8/BF16 & Real-ESRGAN 2x engines.
2. Paste `KISSSHOT_TASK_HEX` from your local `kiss-shot panel` into **Cell 2**:
   - **Automated Queue Daemon (Recommended)**: Paste the daemon token once (`🛰️ 复制远端自动接单守护口令`). Your local desktop console can then dispatch encrypted jobs (`AAD_QUEUE`), reference images (`Img2Img`), painted `Inpainting` masks, `MOSS` spatial BBox layouts, and `I2V` keyframes continuously without switching windows.
   - **Single-Shot Mode**: Paste a one-off task token (`🔒 手动模式`) to run a single inference job.
3. Outputs are packed purely in memory (`BytesIO` / `/dev/shm`) into a double-AEAD archive (`AES-256-GCM` outer `AAD_ARCHIVE` envelope + `256-bit WZ_AES` `ZIP_STORED` inner archive with recipe metadata) and **burned immediately after local decryption**.

In [ ]:
# === Cell 1: Passphrase Unlock, History Scrubbing, TorchAO FP8/BF16 & Double-AEAD Engine ===
import subprocess, sys, os, io, gc, json, re, math, hashlib, shutil, sqlite3, time, secrets, tempfile, threading, importlib.util, getpass

# Prompt immediately at 0.1s before installing packages so window focus is never lost
SESSION_PASSPHRASE = getpass.getpass("🔑 Enter your Kiss-Shot session passphrase (in-memory only): ").strip()
assert SESSION_PASSPHRASE, "Passphrase cannot be empty."
print("Passphrase locked in memory. Installing/verifying inference & cryptography dependencies...")

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def scrub_ipython_history() -> None:
    try:
        ip = get_ipython()  # noqa: F821
        if ip and getattr(ip, "history_manager", None):
            hm = ip.history_manager
            hm.enabled = False
            if getattr(hm, "db", None):
                try:
                    hm.db.execute("DELETE FROM history")
                    hm.db.commit()
                except Exception:
                    pass
        if ip and hasattr(ip, "user_ns"):
            ns = ip.user_ns
            for k in ("In", "_ih"):
                arr = ns.get(k)
                if isinstance(arr, list):
                    for idx in range(len(arr)):
                        if "KISSSHOT_" in str(arr[idx]) or "_HEX" in str(arr[idx]):
                            arr[idx] = "# [cleared]"
            for out_k in ("Out", "_oh"):
                od = ns.get(out_k)
                if isinstance(od, dict):
                    for k in list(od.keys()):
                        if "KISSSHOT_" in str(od[k]) or "_HEX" in str(od[k]):
                            od.pop(k, None)
    except Exception:
        pass
    hist_db = os.path.expanduser("~/.ipython/profile_default/history.sqlite")
    if os.path.exists(hist_db):
        try:
            con = sqlite3.connect(hist_db)
            con.execute("DELETE FROM history")
            con.commit()
            con.execute("VACUUM")
            con.close()
        except Exception:
            pass

scrub_ipython_history()

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "diffusers>=0.35.0", "transformers>=4.49.0", "accelerate>=1.4.0", "sentencepiece",
    "ftfy", "peft", "spandrel", "imageio[ffmpeg]", "cryptography", "pyzipper",
    "huggingface_hub", "hf_transfer", "bitsandbytes"
])
if importlib.util.find_spec("torchao") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "torchao"])
    except Exception:
        pass

import numpy as np
import torch, pyzipper
from PIL import Image, ImageFilter
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    gpu_name = torch.cuda.get_device_name(0)
    cuda_cc = torch.cuda.get_device_capability(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    print(f"✅ Worker ready on GPU: {gpu_name} ({vram_gb} GB VRAM, SM {cuda_cc[0]}.{cuda_cc[1]})")
else:
    cuda_cc, vram_gb = (0, 0), 0.0
    print("⚠️ Running on CPU; attach a CUDA GPU runtime for accelerated generation.")

KDF_ITERATIONS = 210_000
SALT_BYTES, NONCE_BYTES, TAG_BYTES = 16, 12, 16
AAD_PAYLOAD = b"kissshot-payload-v2"
AAD_QUEUE = b"kissshot-queue-v2"
AAD_BLOB = b"kissshot-blob-v2"
AAD_ARCHIVE = b"kissshot-archive-v2"
MAX_TTL_SECONDS = 86_400

_SAFE_STAGING_RE = re.compile(r"^_staging/(?:ref|mask)_[0-9a-f]{12}\.bin$")
_SAFE_QUEUE_RE = re.compile(r"^_queue/job_[0-9]{9,12}_[0-9a-f]{8}\.bin$")
_PIPE_CACHE = {"sig": None, "pipe": None, "upscaler": None}
_TORCHAO_CAPS = {"checked": False, "dyn_fp8": None, "wo_cfg": None}
_ENSURED_REPOS: set[str] = set()

def derive_key(pwd: str, salt: bytes) -> bytes:
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=KDF_ITERATIONS)
    return kdf.derive(pwd.encode("utf-8"))

def decrypt_payload_hex(hex_str: str, pwd: str, aad: bytes = AAD_PAYLOAD) -> dict:
    raw = bytes.fromhex("".join(hex_str.split()))
    if len(raw) <= SALT_BYTES + NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated ciphertext")
    pt = AESGCM(derive_key(pwd, raw[:SALT_BYTES])).decrypt(
        raw[SALT_BYTES:SALT_BYTES + NONCE_BYTES],
        raw[SALT_BYTES + NONCE_BYTES:],
        aad,
    )
    parsed = json.loads(pt.decode("utf-8"))
    if isinstance(parsed, dict) and parsed.get("_ks_v") == 2 and "d" in parsed:
        iat = int(parsed.get("_ks_iat", 0) or 0)
        if not iat or abs(time.time() - iat) > MAX_TTL_SECONDS:
            raise ValueError("Expired task payload (>24h)")
        return parsed["d"]
    return parsed

def encrypt_bytes_aes(raw: bytes, pwd: str, aad: bytes = AAD_BLOB) -> bytes:
    salt, nonce = secrets.token_bytes(SALT_BYTES), secrets.token_bytes(NONCE_BYTES)
    ct = AESGCM(derive_key(pwd, salt)).encrypt(nonce, raw, aad)
    return salt + nonce + ct

def decrypt_bytes_aes(enc: bytes, pwd: str, aad: bytes = AAD_BLOB) -> bytes:
    if len(enc) <= SALT_BYTES + NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated encrypted blob")
    return AESGCM(derive_key(pwd, enc[:SALT_BYTES])).decrypt(
        enc[SALT_BYTES:SALT_BYTES + NONCE_BYTES],
        enc[SALT_BYTES + NONCE_BYTES:],
        aad,
    )

def decrypt_queue_job(enc: bytes, pwd: str) -> dict:
    pt = decrypt_bytes_aes(enc, pwd, aad=AAD_QUEUE)
    parsed = json.loads(pt.decode("utf-8"))
    if not isinstance(parsed, dict) or parsed.get("_ks_v") != 2 or "d" not in parsed:
        raise ValueError("Malformed queue job envelope")
    iat = int(parsed.get("_ks_iat", 0) or 0)
    if not iat or abs(time.time() - iat) > MAX_TTL_SECONDS:
        raise ValueError("Expired queue job (>24h)")
    return parsed["d"]

def pack_encrypted_zip(items: list, pwd: str, meta: dict) -> bytes:
    buf = io.BytesIO()
    pwd_b = pwd.encode("utf-8")
    manifest = {"version": 2, "created_at": int(time.time()), "files": [], "meta": meta}
    with pyzipper.AESZipFile(buf, "w", compression=pyzipper.ZIP_STORED, encryption=pyzipper.WZ_AES) as zf:
        zf.setpassword(pwd_b)
        zf.setencryption(pyzipper.WZ_AES, nbits=256)
        for i, (ext, data) in enumerate(items):
            inner = f"chunk_{i:03d}.dat"
            zf.writestr(inner, data)
            manifest["files"].append({"inner": inner, "ext": ext.lstrip(".") or "bin"})
        zf.writestr("manifest.json", json.dumps(manifest, ensure_ascii=False).encode("utf-8"))
    return encrypt_bytes_aes(buf.getvalue(), pwd, aad=AAD_ARCHIVE)

def _get_torchao_caps() -> dict:
    if _TORCHAO_CAPS["checked"] or not torch.cuda.is_available():
        return _TORCHAO_CAPS
    _TORCHAO_CAPS["checked"] = True
    try:
        import torchao.quantization as tq
        quantize_ = tq.quantize_
        if cuda_cc >= (8, 9) and hasattr(tq, "Float8DynamicActivationFloat8WeightConfig"):
            try:
                probe = torch.nn.Linear(1024, 1024, bias=False, device="cuda", dtype=torch.bfloat16)
                cfg_dyn = tq.Float8DynamicActivationFloat8WeightConfig()
                quantize_(probe, cfg_dyn)
                out = probe(torch.ones((1, 32, 1024), device="cuda", dtype=torch.bfloat16))
                if out.shape == (1, 32, 1024):
                    _TORCHAO_CAPS["dyn_fp8"] = cfg_dyn
                del probe, out
            except Exception:
                pass
        for cls_name in ("Float8WeightOnlyConfig", "Int8WeightOnlyConfig"):
            cfg_cls = getattr(tq, cls_name, None)
            if cfg_cls is None:
                continue
            try:
                probe = torch.nn.Linear(1024, 1024, bias=True, device="cuda", dtype=torch.bfloat16)
                cfg_wo = cfg_cls()
                quantize_(probe, cfg_wo)
                out = probe(torch.ones((1, 32, 1024), device="cuda", dtype=torch.bfloat16))
                if out.shape == (1, 32, 1024):
                    _TORCHAO_CAPS["wo_cfg"] = cfg_wo
                    del probe, out
                    break
                del probe, out
            except Exception:
                pass
        torch.cuda.empty_cache()
    except Exception:
        pass
    return _TORCHAO_CAPS

def _dit_linear_filter(mod: torch.nn.Module, _fqn: str) -> bool:
    return (
        isinstance(mod, torch.nn.Linear)
        and mod.in_features >= 1024
        and mod.out_features >= 1024
        and (mod.in_features % 16 == 0)
        and (mod.out_features % 16 == 0)
    )

def _stream_dit_to_cuda(transformer: torch.nn.Module, allow_fp8_tc: bool = True, need_shrink: bool = False) -> str:
    caps = _get_torchao_caps()
    q_cfg = None
    mode_tag = "BF16 full-VRAM"
    if allow_fp8_tc and caps["dyn_fp8"] is not None:
        q_cfg = caps["dyn_fp8"]
        mode_tag = "FP8 TensorCore full-VRAM"
    elif need_shrink and caps["wo_cfg"] is not None:
        q_cfg = caps["wo_cfg"]
        mode_tag = "8-bit weight-only full-VRAM"
    if q_cfg is not None:
        from torchao.quantization import quantize_
        hooked = False
        for attr in ("layers", "transformer_blocks", "single_transformer_blocks"):
            blks = getattr(transformer, attr, None)
            if isinstance(blks, torch.nn.ModuleList):
                for blk in blks:
                    blk.to("cuda")
                    quantize_(blk, q_cfg, filter_fn=_dit_linear_filter)
                hooked = True
        transformer.to("cuda")
        if not hooked:
            quantize_(transformer, q_cfg, filter_fn=_dit_linear_filter)
        torch.cuda.empty_cache()
        return mode_tag
    if need_shrink and hasattr(transformer, "enable_layerwise_casting"):
        transformer.enable_layerwise_casting(storage_dtype=torch.float8_e4m3fn, compute_dtype=torch.bfloat16)
        transformer.to("cuda")
        return "FP8 layerwise full-VRAM"
    transformer.to("cuda")
    return mode_tag

def _load_text_encoder_vram(model_cls, load_path: str, subfolder: str | None = None):
    kw = {"local_files_only": True}
    if subfolder:
        kw["subfolder"] = subfolder
    if vram_gb >= 34.0:
        enc = model_cls.from_pretrained(load_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, **kw).eval()
        if hasattr(enc, "visual"):
            enc.visual = None
        return enc.to("cuda")
    caps = _get_torchao_caps()
    if caps["wo_cfg"] is not None:
        from torchao.quantization import quantize_
        enc = model_cls.from_pretrained(load_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, **kw).eval()
        if hasattr(enc, "visual"):
            enc.visual = None
        enc.to("cuda")
        quantize_(enc, caps["wo_cfg"], filter_fn=lambda m, _: isinstance(m, torch.nn.Linear) and m.in_features >= 512)
        torch.cuda.empty_cache()
        return enc
    try:
        from transformers import BitsAndBytesConfig
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=["visual"],
        )
        enc = model_cls.from_pretrained(load_path, quantization_config=bnb_cfg, device_map={"": 0}, low_cpu_mem_usage=True, **kw).eval()
        if hasattr(enc, "visual"):
            enc.visual = None
        return enc
    except Exception:
        enc = model_cls.from_pretrained(load_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, **kw).eval()
        if hasattr(enc, "visual"):
            enc.visual = None
        return enc.to("cuda")

def _upscale_pil_2x(img: Image.Image) -> Image.Image:
    from spandrel import ModelLoader
    if _PIPE_CACHE["upscaler"] is None:
        up_dir = "/tmp/kissshot_ckpt/_upscaler"
        os.makedirs(up_dir, exist_ok=True)
        up_p = hf_hub_download(repo_id="ai-forever/Real-ESRGAN", filename="RealESRGAN_x2.pth", local_dir=up_dir, token=False)
        shutil.rmtree(os.path.join(up_dir, ".cache"), ignore_errors=True)
        _PIPE_CACHE["upscaler"] = ModelLoader().load_from_file(up_p).model.eval().to("cuda", dtype=torch.float16)
    net = _PIPE_CACHE["upscaler"]
    arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
    ten = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    _, _, h, w = ten.shape
    scale, tile, overlap = 2, 512, 32
    out = torch.zeros((1, 3, h * scale, w * scale), dtype=torch.float32)
    wgt = torch.zeros((1, 1, h * scale, w * scale), dtype=torch.float32)
    with torch.no_grad():
        for y in range(0, h, tile - overlap):
            for x in range(0, w, tile - overlap):
                y1, x1 = min(h, y + tile), min(w, x + tile)
                y0, x0 = max(0, y1 - tile), max(0, x1 - tile)
                patch = ten[:, :, y0:y1, x0:x1].to("cuda", dtype=torch.float16, non_blocking=True)
                sr = net(patch).float().cpu().clamp(0.0, 1.0)
                out[:, :, y0 * scale:y1 * scale, x0 * scale:x1 * scale] += sr
                wgt[:, :, y0 * scale:y1 * scale, x0 * scale:x1 * scale] += 1.0
    res = (out / wgt.clamp_min(1.0)).squeeze(0).permute(1, 2, 0).numpy()
    return Image.fromarray((res * 255.0).round().astype(np.uint8))

def _moss_img2img_or_inpaint(pipe, ref_img: Image.Image, mask_img: Image.Image | None, prompt_str: str, neg_str: str,
                            width: int, height: int, steps: int, cfg_val: float, strength: float, gen: torch.Generator) -> Image.Image:
    from diffusers.utils.torch_utils import randn_tensor
    ref_resized = ref_img.convert("RGB").resize((width, height), Image.Resampling.LANCZOS)
    device = pipe._execution_device
    prompt_embeds, prompt_mask = pipe.encode_prompt(prompt_str, 4096)
    positive, positive_mask = prompt_embeds.to(device), prompt_mask.to(device)
    do_cfg = cfg_val != 1.0
    uncond_skip = () if neg_str else (22,)
    if do_cfg:
        neg_embeds, neg_mask = pipe.encode_prompt([neg_str], 4096)
        negative, negative_mask = neg_embeds.to(device), neg_mask.to(device)

    x_np = (np.asarray(ref_resized, dtype=np.float32) / 127.5) - 1.0
    x_in = torch.from_numpy(x_np).permute(2, 0, 1).unsqueeze(0).to(device=device, dtype=pipe.vae.dtype)
    with torch.no_grad():
        z_raw = pipe.vae.encode(x_in).latent_dist.mode()
        b, c_v, h2, w2 = z_raw.shape
        h_l, w_l = h2 // 2, w2 // 2
        z_pack = z_raw.reshape(b, c_v, h_l, 2, w_l, 2).permute(0, 1, 3, 5, 2, 4).reshape(b, c_v * 4, h_l, w_l)
        bn = pipe.vae.bn
        eps = pipe.vae.config.batch_norm_eps
        z_0 = ((z_pack - bn.running_mean.to(z_pack).view(1, -1, 1, 1)) / (bn.running_var + eps).sqrt().to(z_pack).view(1, -1, 1, 1)).float()

    mask_lat, feather_mask = None, None
    if mask_img is not None:
        m_resized = mask_img.convert("L").resize((width, height), Image.Resampling.BILINEAR)
        feather_mask = m_resized.filter(ImageFilter.GaussianBlur(radius=6))
        m_small = feather_mask.resize((w_l, h_l), Image.Resampling.BILINEAR)
        mask_lat = torch.from_numpy(np.asarray(m_small, dtype=np.float32) / 255.0).view(1, 1, h_l, w_l).to(device=device, dtype=torch.float32)

    pipe.scheduler.set_timesteps(steps, device=device)
    total_t = len(pipe.scheduler.timesteps)
    t_idx = min(total_t - 1, max(0, int(round((1.0 - strength) * (total_t - 1)))))
    pipe.scheduler.timesteps = pipe.scheduler.timesteps[t_idx:]
    pipe.scheduler.sigmas = pipe.scheduler.sigmas[t_idx:]
    pipe.scheduler._step_index = None

    noise = randn_tensor(z_0.shape, generator=gen, device=device, dtype=torch.float32)
    sigma0 = pipe.scheduler.sigmas[0].to(z_0)
    latents = (1.0 - sigma0) * z_0 + sigma0 * noise

    with torch.no_grad():
        for s_i, timestep in enumerate(pipe.progress_bar(pipe.scheduler.timesteps)):
            pred = pipe.transformer(latents, timestep, positive, positive_mask).sample.float()
            if do_cfg:
                sk = {"skip_layers": uncond_skip} if uncond_skip else {}
                uncond = pipe.transformer(latents, timestep, negative, negative_mask, **sk).sample.float()
                pred = uncond + cfg_val * (pred - uncond)
            latents = pipe.scheduler.step(pred, timestep, latents, return_dict=False)[0]
            if mask_lat is not None and (s_i + 1) < len(pipe.scheduler.sigmas):
                sig_next = pipe.scheduler.sigmas[s_i + 1].to(z_0)
                ref_noisy = (1.0 - sig_next) * z_0 + sig_next * noise
                latents = mask_lat * latents + (1.0 - mask_lat) * ref_noisy

        z_out = latents.to(dtype=pipe.vae.dtype)
        z_out = z_out * (bn.running_var + eps).sqrt().to(z_out).view(1, -1, 1, 1) + bn.running_mean.to(z_out).view(1, -1, 1, 1)
        b, c, h_o, w_o = z_out.shape
        z_out = z_out.reshape(b, c // 4, 2, 2, h_o, w_o).permute(0, 1, 4, 2, 5, 3).reshape(b, c // 4, 2 * h_o, 2 * w_o)
        decoded = pipe.vae.decode(z_out, return_dict=False)[0].clamp(-1, 1)
        out_pil = pipe.image_processor.postprocess(decoded, output_type="pil")[0]
    pipe.maybe_free_model_hooks()
    if feather_mask is not None:
        out_pil = Image.composite(out_pil, ref_resized, feather_mask)
    return out_pil

def _pull_and_burn_staging_img(mailbox_repo: str, staging_rel: str, hf_tok: str) -> Image.Image:
    if not _SAFE_STAGING_RE.fullmatch(staging_rel):
        raise ValueError(f"Unsafe staging path rejected: {staging_rel!r}")
    shm = "/dev/shm" if os.path.isdir("/dev/shm") else None
    with tempfile.TemporaryDirectory(dir=shm) as td:
        enc_p = hf_hub_download(repo_id=mailbox_repo, filename=staging_rel, repo_type="model", token=hf_tok, local_dir=td)
        with open(enc_p, "rb") as f_enc:
            raw_b = decrypt_bytes_aes(f_enc.read(), SESSION_PASSPHRASE, aad=AAD_BLOB)
    def _bg_burn(rel_p: str, repo_p: str, tok_p: str) -> None:
        try:
            HfApi(token=tok_p).delete_file(path_in_repo=rel_p, repo_id=repo_p, repo_type="model", commit_message="burn staging")
        except Exception:
            pass
    threading.Thread(target=_bg_burn, args=(staging_rel, mailbox_repo, hf_tok), daemon=True).start()
    return Image.open(io.BytesIO(raw_b))

In [ ]:
# === Cell 2: Paste KISSSHOT_TASK_HEX (Automated Queue Daemon or Single-Shot Task) ===
KISSSHOT_TASK_HEX = ""

def _execute_media_task(task: dict) -> None:
    assert task.get("type") == "kissshot_media_infer", "Invalid media task payload type."
    import inspect
    target_repo = str(task["src_repo"])
    preset_key = str(task.get("preset_key", "preset"))
    raw_arch = str(task["arch"])
    variant = task.get("variant")
    skip_root = bool(task.get("skip_root", False))
    hf_token = str(task["hf_token"])
    mailbox_repo = str(task["mailbox_repo"])
    prompt_str = str(task["prompt"])
    neg_str = str(task.get("negative_prompt", ""))
    seed = int(task["seed"]) if int(task["seed"]) >= 0 else secrets.randbits(31)
    width, height = (int(task["width"]) // 16) * 16, (int(task["height"]) // 16) * 16
    steps, cfg_val = int(task["steps"]), float(task["cfg"])
    batch_cnt = max(1, int(task.get("batch", 1)))
    frames_req = int(task.get("frames", 81))
    strength = max(0.05, min(1.0, float(task.get("strength", 0.65))))
    upscale_2x = bool(task.get("upscale_2x", False))
    ref_staging = str(task.get("ref_staging_blob", ""))
    mask_staging = str(task.get("mask_staging_blob", ""))
    del task

    # Dual-mode Wan TI2V switches to ImageToVideo pipeline when a reference keyframe is supplied
    eff_arch = ("wan_i2v" if ref_staging else "wan_t2v") if raw_arch == "wan_ti2v" else raw_arch
    api = HfApi(token=hf_token)
    repo_sig = hashlib.sha256(f"{target_repo}|{variant}|{skip_root}|{eff_arch}".encode("utf-8")).hexdigest()
    ckpt_hash = hashlib.sha256(f"{target_repo}|{variant}|{skip_root}".encode("utf-8")).hexdigest()[:12]
    ckpt_dir = f"/tmp/kissshot_ckpt/{ckpt_hash}"

    try:
        if _PIPE_CACHE["sig"] != repo_sig:
            if _PIPE_CACHE["pipe"] is not None:
                del _PIPE_CACHE["pipe"]
                _PIPE_CACHE["pipe"] = None
                _PIPE_CACHE["sig"] = None
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            if not os.path.isfile(os.path.join(ckpt_dir, ".ready")):
                os.makedirs(ckpt_dir, exist_ok=True)
                ignore_pats = ["*.md", "README*", ".git*"]
                if skip_root:
                    try:
                        tree = [x.path for x in HfApi(token=False).list_repo_tree(repo_id=target_repo, repo_type="model", recursive=True) if hasattr(x, "size")]
                        pull_tok = False
                    except Exception:
                        tree = [x.path for x in HfApi(token=hf_token).list_repo_tree(repo_id=target_repo, repo_type="model", recursive=True) if hasattr(x, "size")]
                        pull_tok = hf_token
                    allow_pats = [p for p in tree if not p.startswith(".git") and not ("/" not in p and p.lower().endswith(".safetensors"))]
                    snapshot_download(repo_id=target_repo, local_dir=ckpt_dir, allow_patterns=allow_pats, token=pull_tok)
                else:
                    try:
                        snapshot_download(repo_id=target_repo, local_dir=ckpt_dir, ignore_patterns=ignore_pats, token=False)
                    except Exception:
                        snapshot_download(repo_id=target_repo, local_dir=ckpt_dir, ignore_patterns=ignore_pats, token=hf_token)
                shutil.rmtree(os.path.join(ckpt_dir, ".cache"), ignore_errors=True)
                open(os.path.join(ckpt_dir, ".ready"), "w").write(ckpt_hash)

            if eff_arch == "moss":
                from transformers import AutoTokenizer, AutoModel
                from diffusers import AutoencoderKLFlux2
                moss_py = os.path.join(ckpt_dir, "pipeline_moss_image.py")
                spec_m = importlib.util.spec_from_file_location("pipeline_moss_image", moss_py)
                moss_mod = importlib.util.module_from_spec(spec_m)
                sys.modules["pipeline_moss_image"] = moss_mod
                spec_m.loader.exec_module(moss_mod)
                text_enc = _load_text_encoder_vram(AutoModel, os.path.join(ckpt_dir, "text_encoder"))
                dit = moss_mod.MOSSImageTransformer.from_pretrained(
                    os.path.join(ckpt_dir, "transformer"), torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, local_files_only=True
                ).eval()
                _stream_dit_to_cuda(dit, allow_fp8_tc=True, need_shrink=(vram_gb < 18.0))
                vae = AutoencoderKLFlux2.from_pretrained(
                    os.path.join(ckpt_dir, "vae"), torch_dtype=torch.float32, low_cpu_mem_usage=True, local_files_only=True
                ).eval().to("cuda")
                sched_dir = os.path.join(ckpt_dir, "scheduler")
                moss_sched = (
                    moss_mod.MOSSImageScheduler.from_pretrained(sched_dir, local_files_only=True)
                    if hasattr(moss_mod.MOSSImageScheduler, "from_pretrained")
                    else moss_mod.MOSSImageScheduler.from_config(sched_dir, local_files_only=True)
                )
                pipe = moss_mod.MOSSImagePipeline(
                    transformer=dit,
                    scheduler=moss_sched,
                    text_encoder=text_enc,
                    tokenizer=AutoTokenizer.from_pretrained(os.path.join(ckpt_dir, "tokenizer"), local_files_only=True),
                    vae=vae,
                )
            elif eff_arch == "sdxl":
                from diffusers import StableDiffusionXLPipeline
                if os.path.isfile(os.path.join(ckpt_dir, "model_index.json")):
                    kw = {"torch_dtype": torch.bfloat16, "use_safetensors": True, "local_files_only": True}
                    if variant:
                        kw["variant"] = variant
                    pipe = StableDiffusionXLPipeline.from_pretrained(ckpt_dir, **kw)
                else:
                    sf_list = sorted([os.path.join(ckpt_dir, x) for x in os.listdir(ckpt_dir) if x.lower().endswith(".safetensors")])
                    pipe = StableDiffusionXLPipeline.from_single_file(sf_list[0], torch_dtype=torch.bfloat16)
                pipe.vae.enable_tiling()
                pipe.to("cuda")
            elif eff_arch == "flux":
                from diffusers import FluxPipeline, FluxTransformer2DModel
                from transformers import T5EncoderModel
                text_enc_2 = _load_text_encoder_vram(T5EncoderModel, ckpt_dir, subfolder="text_encoder_2")
                dit = FluxTransformer2DModel.from_pretrained(
                    ckpt_dir, subfolder="transformer", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, local_files_only=True
                )
                _stream_dit_to_cuda(dit, allow_fp8_tc=True, need_shrink=(vram_gb < 38.0))
                pipe = FluxPipeline.from_pretrained(
                    ckpt_dir, text_encoder_2=text_enc_2, transformer=dit, torch_dtype=torch.bfloat16, local_files_only=True
                )
                pipe.vae.enable_slicing()
                pipe.vae.enable_tiling()
                pipe.to("cuda")
            elif eff_arch == "auto":
                from diffusers import DiffusionPipeline
                pipe = DiffusionPipeline.from_pretrained(ckpt_dir, torch_dtype=torch.bfloat16, local_files_only=True)
                if vram_gb >= 22.0:
                    try:
                        pipe.to("cuda")
                    except RuntimeError:
                        torch.cuda.empty_cache()
                        pipe.enable_model_cpu_offload()
                else:
                    pipe.enable_model_cpu_offload()
            elif eff_arch == "cogvideox_i2v":
                from diffusers import CogVideoXImageToVideoPipeline
                from transformers import T5EncoderModel
                text_enc = _load_text_encoder_vram(T5EncoderModel, ckpt_dir, subfolder="text_encoder")
                pipe = CogVideoXImageToVideoPipeline.from_pretrained(
                    ckpt_dir, text_encoder=text_enc, torch_dtype=torch.bfloat16, local_files_only=True
                )
                pipe.vae.enable_slicing()
                pipe.vae.enable_tiling()
                if vram_gb >= 20.0:
                    pipe.to("cuda")
                else:
                    pipe.enable_model_cpu_offload()
            elif eff_arch == "hunyuan15_t2v":
                from diffusers import HunyuanVideo15Pipeline
                pipe = HunyuanVideo15Pipeline.from_pretrained(ckpt_dir, torch_dtype=torch.bfloat16, local_files_only=True)
                pipe.vae.enable_tiling()
                if vram_gb >= 38.0:
                    pipe.enable_model_cpu_offload()
                else:
                    pipe.enable_sequential_cpu_offload()
            elif eff_arch == "wan_t2v":
                from diffusers import WanPipeline, AutoencoderKLWan
                vae = AutoencoderKLWan.from_pretrained(ckpt_dir, subfolder="vae", torch_dtype=torch.float32, local_files_only=True)
                pipe = WanPipeline.from_pretrained(ckpt_dir, vae=vae, torch_dtype=torch.bfloat16, local_files_only=True)
                if vram_gb >= 38.0:
                    pipe.enable_model_cpu_offload()
                else:
                    pipe.enable_sequential_cpu_offload()
            elif eff_arch == "wan_i2v":
                from diffusers import WanImageToVideoPipeline, AutoencoderKLWan
                vae = AutoencoderKLWan.from_pretrained(ckpt_dir, subfolder="vae", torch_dtype=torch.float32, local_files_only=True)
                pipe = WanImageToVideoPipeline.from_pretrained(ckpt_dir, vae=vae, torch_dtype=torch.bfloat16, local_files_only=True)
                if vram_gb >= 38.0:
                    pipe.enable_model_cpu_offload()
                else:
                    pipe.enable_sequential_cpu_offload()
            else:
                raise ValueError(f"Unsupported architecture: {eff_arch}")

            _PIPE_CACHE["sig"] = repo_sig
            _PIPE_CACHE["pipe"] = pipe

        pipe = _PIPE_CACHE["pipe"]
        out_blobs = []
        t0 = time.time()

        if eff_arch in ("moss", "sdxl", "flux", "auto"):
            ref_img = _pull_and_burn_staging_img(mailbox_repo, ref_staging, hf_token).convert("RGB") if ref_staging else None
            mask_img = _pull_and_burn_staging_img(mailbox_repo, mask_staging, hf_token).convert("L") if (ref_img and mask_staging) else None
            for idx in range(batch_cnt):
                gen = torch.Generator(device="cpu").manual_seed(seed + idx)
                if ref_img is not None and eff_arch == "moss":
                    img = _moss_img2img_or_inpaint(pipe, ref_img, mask_img, prompt_str, neg_str, width, height, steps, cfg_val, strength, gen)
                elif ref_img is not None:
                    from diffusers import AutoPipelineForImage2Image, AutoPipelineForInpainting
                    ref_r = ref_img.resize((width, height), Image.Resampling.LANCZOS)
                    if mask_img is not None:
                        mask_r = mask_img.resize((width, height), Image.Resampling.BILINEAR)
                        feather_m = mask_r.filter(ImageFilter.GaussianBlur(radius=6))
                        try:
                            sub_p = AutoPipelineForInpainting.from_pipe(pipe)
                            sp_sig = set(inspect.signature(sub_p.__call__).parameters.keys())
                            ikw = dict(prompt=prompt_str, image=ref_r, mask_image=mask_r, width=width, height=height,
                                       strength=strength, num_inference_steps=steps, guidance_scale=cfg_val, generator=gen)
                            if "negative_prompt" in sp_sig and neg_str:
                                ikw["negative_prompt"] = neg_str
                            img = sub_p(**ikw).images[0]
                        except Exception:
                            sub_p = AutoPipelineForImage2Image.from_pipe(pipe)
                            sp_sig = set(inspect.signature(sub_p.__call__).parameters.keys())
                            ikw = dict(prompt=prompt_str, image=ref_r, strength=strength, num_inference_steps=steps, guidance_scale=cfg_val, generator=gen)
                            if "negative_prompt" in sp_sig and neg_str:
                                ikw["negative_prompt"] = neg_str
                            img = sub_p(**ikw).images[0]
                        img = Image.composite(img.resize((width, height), Image.Resampling.LANCZOS), ref_r, feather_m)
                    else:
                        sub_p = AutoPipelineForImage2Image.from_pipe(pipe)
                        sp_sig = set(inspect.signature(sub_p.__call__).parameters.keys())
                        ikw = dict(prompt=prompt_str, image=ref_r, strength=strength, num_inference_steps=steps, guidance_scale=cfg_val, generator=gen)
                        if "width" in sp_sig:
                            ikw["width"], ikw["height"] = width, height
                        if "negative_prompt" in sp_sig and neg_str:
                            ikw["negative_prompt"] = neg_str
                        img = sub_p(**ikw).images[0]
                else:
                    sig_p = set(inspect.signature(pipe.__call__).parameters.keys())
                    kw = dict(prompt=prompt_str, width=width, height=height, num_inference_steps=steps, guidance_scale=cfg_val, generator=gen)
                    if "negative_prompt" in sig_p and neg_str:
                        kw["negative_prompt"] = neg_str
                        if eff_arch == "moss" and "uncond_skip_layers" in sig_p:
                            kw["uncond_skip_layers"] = ()
                    img = pipe(**kw).images[0]

                if upscale_2x:
                    img = _upscale_pil_2x(img)
                mem = io.BytesIO()
                img.save(mem, format="PNG", compress_level=1)
                out_blobs.append(("png", mem.getvalue()))
        else:
            from diffusers.utils import export_to_video
            frames_cnt = ((max(17, frames_req) - 1) // 4) * 4 + 1
            gen = torch.Generator(device="cpu").manual_seed(seed)
            sig_p = set(inspect.signature(pipe.__call__).parameters.keys())
            if eff_arch in ("wan_i2v", "cogvideox_i2v"):
                if not ref_staging:
                    raise ValueError("Image-to-Video architecture requires an encrypted reference image.")
                ref_img = _pull_and_burn_staging_img(mailbox_repo, ref_staging, hf_token).convert("RGB")
                if eff_arch == "cogvideox_i2v":
                    ref_img = ref_img.resize((width, height), Image.Resampling.LANCZOS)
                    vkw = dict(image=ref_img, prompt=prompt_str, height=height, width=width,
                               num_frames=frames_cnt, guidance_scale=cfg_val, num_inference_steps=steps, generator=gen)
                    if "negative_prompt" in sig_p and neg_str:
                        vkw["negative_prompt"] = neg_str
                    if "use_dynamic_cfg" in sig_p:
                        vkw["use_dynamic_cfg"] = True
                    vid_frames = pipe(**vkw).frames[0]
                else:
                    mod_val = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]
                    max_area = width * height
                    aspect = ref_img.height / ref_img.width
                    height = max(mod_val, (round(math.sqrt(max_area * aspect)) // mod_val) * mod_val)
                    width = max(mod_val, (round(math.sqrt(max_area / aspect)) // mod_val) * mod_val)
                    ref_img = ref_img.resize((width, height), Image.Resampling.LANCZOS)
                    vid_frames = pipe(
                        image=ref_img, prompt=prompt_str, negative_prompt=neg_str, height=height, width=width,
                        num_frames=frames_cnt, guidance_scale=cfg_val, num_inference_steps=steps, generator=gen
                    ).frames[0]
            else:
                vkw = dict(prompt=prompt_str, num_frames=frames_cnt, num_inference_steps=steps, generator=gen)
                if "height" in sig_p:
                    vkw["height"], vkw["width"] = height, width
                if "guidance_scale" in sig_p:
                    vkw["guidance_scale"] = cfg_val
                if "negative_prompt" in sig_p and neg_str:
                    vkw["negative_prompt"] = neg_str
                vid_frames = pipe(**vkw).frames[0]

            shm = "/dev/shm" if os.path.isdir("/dev/shm") else tempfile.gettempdir()
            tmp_mp4 = os.path.join(shm, f"ks_{secrets.token_hex(6)}.mp4")
            out_fps = 24 if (eff_arch == "hunyuan15_t2v" or raw_arch == "wan_ti2v") else 16
            try:
                export_to_video(vid_frames, tmp_mp4, fps=out_fps)
                with open(tmp_mp4, "rb") as vf:
                    out_blobs.append(("mp4", vf.read()))
            finally:
                if os.path.exists(tmp_mp4):
                    os.remove(tmp_mp4)

        is_png = bool(out_blobs and out_blobs[0][0] == "png")
        recipe_meta = {
            "seed": seed,
            "model": preset_key,
            "preset": preset_key,
            "prompt": prompt_str,
            "negative_prompt": neg_str,
            "width": width * (2 if (upscale_2x and is_png) else 1),
            "height": height * (2 if (upscale_2x and is_png) else 1),
            "steps": steps,
            "cfg": cfg_val,
            "frames": frames_req,
            "strength": strength,
            "upscale_2x": upscale_2x,
        }
        enc_zip = pack_encrypted_zip(out_blobs, SESSION_PASSPHRASE, meta=recipe_meta)
        del out_blobs
        remote_blob = f"_inbox/pkt_{int(time.time())}_{secrets.token_hex(4)}.bin"
        if mailbox_repo not in _ENSURED_REPOS:
            api.create_repo(repo_id=mailbox_repo, repo_type="model", private=True, exist_ok=True)
            _ENSURED_REPOS.add(mailbox_repo)
        api.upload_file(path_or_fileobj=enc_zip, path_in_repo=remote_blob, repo_id=mailbox_repo, repo_type="model", commit_message="sync artifact")
        print(f"✅ Done in {time.time() - t0:.1f}s (seed={seed}). Double-AEAD archive uploaded to {mailbox_repo}/{remote_blob}.")
    finally:
        prompt_str = ""
        neg_str = ""
        hf_token = ""
        del api

def _run_queue_daemon(cfg: dict) -> None:
    hf_tok = str(cfg["hf_token"])
    mailbox_repo = str(cfg["mailbox_repo"])
    del cfg
    api = HfApi(token=hf_tok)
    if mailbox_repo not in _ENSURED_REPOS:
        api.create_repo(repo_id=mailbox_repo, repo_type="model", private=True, exist_ok=True)
        _ENSURED_REPOS.add(mailbox_repo)
    idle_since = time.time()
    max_idle_sec = 5400
    print("🛰️ Kiss-Shot ciphertext queue daemon listening (1.5s poll, 90m idle auto-stop). Dispatch jobs directly from your local desktop console!")
    try:
        while True:
            if time.time() - idle_since > max_idle_sec:
                print("Idle timeout (90m) reached; exiting queue daemon cleanly.")
                break
            try:
                files = api.list_repo_files(repo_id=mailbox_repo, repo_type="model")
                q_files = sorted([f for f in files if _SAFE_QUEUE_RE.fullmatch(f)])
            except Exception as poll_e:
                print("Transient poll error, retrying in 3s:", str(poll_e)[:80])
                time.sleep(3)
                continue
            if not q_files:
                time.sleep(1.5)
                continue
            for qf in q_files:
                idle_since = time.time()
                print(f"\n[Queue] Dequeuing encrypted job {qf} ...")
                shm = "/dev/shm" if os.path.isdir("/dev/shm") else None
                with tempfile.TemporaryDirectory(dir=shm) as td:
                    lp = hf_hub_download(repo_id=mailbox_repo, filename=qf, repo_type="model", token=hf_tok, local_dir=td)
                    with open(lp, "rb") as jf:
                        job_bytes = jf.read()
                def _bg_del(rel_q: str, repo_q: str, tok_q: str) -> None:
                    try:
                        HfApi(token=tok_q).delete_file(path_in_repo=rel_q, repo_id=repo_q, repo_type="model", commit_message="dequeue job")
                    except Exception:
                        pass
                threading.Thread(target=_bg_del, args=(qf, mailbox_repo, hf_tok), daemon=True).start()
                try:
                    job_obj = decrypt_queue_job(job_bytes, SESSION_PASSPHRASE)
                    _execute_media_task(job_obj)
                except Exception as job_err:
                    print(f"❌ Job execution error: {job_err}")
                finally:
                    job_obj = None
                    gc.collect()
                idle_since = time.time()
    except KeyboardInterrupt:
        print("Queue daemon stopped.")
    finally:
        hf_tok = ""
        del api

def _run_entry(raw_hex: str) -> None:
    assert raw_hex.strip(), "Paste KISSSHOT_TASK_HEX (daemon or single-shot) from your local console first."
    obj = decrypt_payload_hex(raw_hex, SESSION_PASSPHRASE)
    kind = obj.get("type")
    if kind == "kissshot_media_daemon":
        _run_queue_daemon(obj)
    elif kind == "kissshot_media_infer":
        _execute_media_task(obj)
    else:
        raise ValueError(f"Unsupported task payload type: {kind!r}")

_tmp_ks_hex = KISSSHOT_TASK_HEX
KISSSHOT_TASK_HEX = ""
scrub_ipython_history()
try:
    _run_entry(_tmp_ks_hex)
finally:
    _tmp_ks_hex = ""
    scrub_ipython_history()
    gc.collect()